In [ ]:
import numpy as np
import cvxpy as cp
import pandas as pd

# ========= 1) Excel 读列 =========
file_path = r"your_data.xlsx"
sheet_name = "IP"
# 同 LP，实际矩阵结构按题目自行组织读取

# ========= 2) 参数模板 =========
params = {
    "c": np.array([3.0, 2.0, 5.0]),                 # 目标系数
    "A_ub": np.array([[2.0, 1.0, 1.0]]),            # 不等式约束矩阵
    "b_ub": np.array([4.0]),                        # 不等式右端
    "A_eq": None,                                   # 等式约束矩阵
    "b_eq": None,                                   # 等式右端
    "var_type": "binary"                            # 'binary'/'integer'
}

n = len(params["c"])
x = cp.Variable(n, boolean=True) if params["var_type"] == "binary" else cp.Variable(n, integer=True)
cons = [params["A_ub"] @ x <= params["b_ub"]]
if params["A_eq"] is not None and params["b_eq"] is not None:
    cons.append(params["A_eq"] @ x == params["b_eq"])

prob = cp.Problem(cp.Maximize(params["c"] @ x), cons)
prob.solve()
print(x.value, prob.value)


In [ ]:
"""
整数规划 0-1 规划

使用方法：
1. 按照下方 TODO 修改 DATA_FILE、列名、参数和输出文件名。
2. 将数据文件放在本脚本同目录，或把 DATA_FILE 改成绝对路径。
3. 运行：python "整数规划 0-1 规划.py"
"""

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import Bounds, LinearConstraint, milp



DATA_FILE = "data.csv"  # TODO: 请填写[数据文件路径]，说明：CSV/Excel 均可；若使用 Excel，请在 load_data 中改为 read_excel。
OUTPUT_FILE = "model_output.csv"  # TODO: 请填写[输出文件名]，说明：保存模型结果，建议保留 .csv 或 .xlsx 后缀。
RANDOM_STATE = 42  # TODO: 请填写[随机种子]，说明：用于复现实验；整数即可。
OBJECTIVE_COEFFICIENTS = [1, 2, 3]  # TODO: 请填写[目标函数系数]，说明：milp 默认最小化。
A_CONSTRAINT = [[1, 1, 1]]  # TODO: 请填写[约束矩阵]，说明：配合上下界表达 lb <= A @ x <= ub。
LB_CONSTRAINT = [1]  # TODO: 请填写[约束下界]，说明：长度等于约束行数。
UB_CONSTRAINT = [2]  # TODO: 请填写[约束上界]，说明：长度等于约束行数。
LB_VARIABLE = [0, 0, 0]  # TODO: 请填写[变量下界]，说明：0-1 规划填 0。
UB_VARIABLE = [1, 1, 1]  # TODO: 请填写[变量上界]，说明：0-1 规划填 1。



REQUIRES_DATA = False  # 参数型模型可不提供数据文件；表格型模型必须提供数据。


def load_data() -> pd.DataFrame:
    """读取用户数据；竞赛时通常把 Excel/CSV 表格整理成一行一个样本。"""
    path = Path(DATA_FILE)
    if not path.exists():
        if not REQUIRES_DATA:
            return pd.DataFrame()
        raise FileNotFoundError(
            f"未找到数据文件 {DATA_FILE}。请先修改 DATA_FILE，或将数据放到脚本同目录。"
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)


def run_model(data: pd.DataFrame) -> None:
    # scipy milp 默认做最小化；若要求最大化，请把目标系数取负。
    constraints = LinearConstraint(A_CONSTRAINT, LB_CONSTRAINT, UB_CONSTRAINT)
    integrality = np.ones(len(OBJECTIVE_COEFFICIENTS))
    res = milp(c=OBJECTIVE_COEFFICIENTS, integrality=integrality, bounds=Bounds(LB_VARIABLE, UB_VARIABLE), constraints=constraints)
    print(res.message)
    print("最优值:", res.fun)
    print("整数决策变量:", res.x)


if __name__ == "__main__":
    df = load_data()
    run_model(df)


# 整数规划 0-1 规划

## 输入说明

- 数据文件：默认读取脚本同目录下的 `data.csv`，也可以在代码顶部把 `DATA_FILE` 改为 `.xlsx` 或绝对路径。
- 数据格式：一般要求“一行一个样本/时刻/方案，一列一个变量/指标”。具体列名需要在代码顶部的 `TODO` 参数区填写。
- 示例：若模型需要特征 `特征1、特征2` 和目标列 `y`，表格可整理为：

| 特征1 | 特征2 | y |
|---:|---:|---:|
| 1.2 | 3.4 | 8.1 |
| 2.0 | 2.8 | 9.0 |

## 输出说明

- 控制台会打印核心结果，例如模型参数、评价指标、最优解、排名或预测值。
- 默认结果保存到代码顶部 `OUTPUT_FILE` 指定的文件。
- 若模型包含图形分析，会额外输出图片文件，例如箱型图 `boxplot.png`。

## 原理通俗解释

整数/0-1 规划 的核心思想是：先把实际问题抽象成可计算的数据结构，再用对应的数学规则寻找“预测值、分类结果、综合得分或最优方案”。代码中已经保留主要计算流程，比赛时重点是把题目数据整理成表格，并把 TODO 参数替换为题目含义一致的列名和约束。

## 适用场景

选址、排班、装箱、投资组合等离散决策。

## 局限性

变量多时求解复杂度高，可能需要专业求解器。

## 使用提示

- 运行前先检查缺失值、异常值和量纲；很多模型对数据尺度敏感。
- 所有 `TODO` 都应结合题目背景填写，不要直接使用示例列名。
- 建模论文中建议同时写明参数来源，例如权重来自 AHP/熵权法，预测步数来自题目要求。
